# 1. Load data

In [121]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

In [122]:
df = pd.read_csv("carclaims.csv")

In [123]:
df.shape

(15420, 33)

In [124]:
df.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange-Claim,NumberOfCars,Year,BasePolicy,FraudFound
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,...,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability,No
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,...,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision,No
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,...,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision,No
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,...,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability,No
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,...,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision,No


# 2. Clean columns names (cleaning)

In [125]:
df.columns = df.columns.str.lower().str.replace('[^a-z0-9]', '_', regex=True)
print(df.columns.tolist())

['month', 'weekofmonth', 'dayofweek', 'make', 'accidentarea', 'dayofweekclaimed', 'monthclaimed', 'weekofmonthclaimed', 'sex', 'maritalstatus', 'age', 'fault', 'policytype', 'vehiclecategory', 'vehicleprice', 'policynumber', 'repnumber', 'deductible', 'driverrating', 'days_policy_accident', 'days_policy_claim', 'pastnumberofclaims', 'ageofvehicle', 'ageofpolicyholder', 'policereportfiled', 'witnesspresent', 'agenttype', 'numberofsuppliments', 'addresschange_claim', 'numberofcars', 'year', 'basepolicy', 'fraudfound']


# 3. Drop Leakage and Useless Columns

In [126]:
drop_cols = [
    'policynumber',
    'repnumber',
    'daysofpolicyclaimed',
    'dayofweekclaimed',
    'monthclaimed',
    'weekofmonthclaimed',
    'days_policy_claim',
]

drop_cols = [c for c in drop_cols if c in df.columns]
df.drop(columns=drop_cols, inplace=True)
print("Remaining shape:", df.shape)

Remaining shape: (15420, 27)


In [127]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   month                 15420 non-null  object
 1   weekofmonth           15420 non-null  int64 
 2   dayofweek             15420 non-null  object
 3   make                  15420 non-null  object
 4   accidentarea          15420 non-null  object
 5   sex                   15420 non-null  object
 6   maritalstatus         15420 non-null  object
 7   age                   15420 non-null  int64 
 8   fault                 15420 non-null  object
 9   policytype            15420 non-null  object
 10  vehiclecategory       15420 non-null  object
 11  vehicleprice          15420 non-null  object
 12  deductible            15420 non-null  int64 
 13  driverrating          15420 non-null  int64 
 14  days_policy_accident  15420 non-null  object
 15  pastnumberofclaims    15420 non-null

# Split Data

In [128]:
y = df["fraudfound"] #target column
x  = df.drop(columns=["fraudfound"])

X_train, X_other, y_train, y_other = train_test_split(
    x, y, test_size=0.3, stratify=y, random_state=510) #70 - 30 split

X_val, X_test, y_val, y_test = train_test_split(
    X_other, y_other, test_size=0.5, stratify=y_other, random_state=510) #15 - 15 split for validation and test

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Training: (10794, 26) (10794,)
Validation: (2313, 26) (2313,)
Test: (2313, 26) (2313,)


# 5. Handle Missing Values & Fix Errors


In [129]:
df[df["age"] == 0]["fraudfound"].value_counts()

fraudfound
No     289
Yes     31
Name: count, dtype: int64

In [130]:
(df["age"] == 0).sum()

np.int64(320)

In [131]:
# Create flag before changing age
X_train["is_age_placeholder"] = (X_train["age"] == 0).astype(int)
X_test["is_age_placeholder"] = (X_test["age"] == 0).astype(int)

# Convert invalid 0 ages to NaN
X_train["age"] = X_train["age"].replace(0, np.nan)
X_test["age"] = X_test["age"].replace(0, np.nan)

# Compute medians from TRAIN only
group_medians = X_train.groupby(["maritalstatus", "vehicleprice"])["age"].median()
global_median = X_train["age"].median()

# Fill TRAIN
train_keys = pd.MultiIndex.from_frame(X_train[["maritalstatus", "vehicleprice"]])
X_train["age"] = X_train["age"].fillna(
    pd.Series(train_keys.map(group_medians), index=X_train.index)
)
X_train["age"] = X_train["age"].fillna(global_median)


In [132]:
# Fill TEST using TRAIN medians
test_keys = pd.MultiIndex.from_frame(X_test[["maritalstatus", "vehicleprice"]])
X_test["age"] = X_test["age"].fillna(
    pd.Series(test_keys.map(group_medians), index=X_test.index)
)
X_test["age"] = X_test["age"].fillna(global_median)

In [133]:
# Check
print((X_train["age"] == 0).sum())
print((X_test["age"] == 0).sum())

0
0


# 6. Feature engineering (new columns)

In [134]:
#Applied separately to X_train, X_val, X_test

# High claim history
X_train["high_claim_history"] = X_train["pastnumberofclaims"].isin(
    ["2 to 4", "more than 4"]
).astype(int)

X_val["high_claim_history"] = X_val["pastnumberofclaims"].isin(
    ["2 to 4", "more than 4"]
).astype(int)

X_test["high_claim_history"] = X_test["pastnumberofclaims"].isin(
    ["2 to 4", "more than 4"]
).astype(int)


# Weekend accident
X_train["weekend_accident"] = X_train["dayofweek"].isin(
    ["Saturday", "Sunday"]
).astype(int)

X_val["weekend_accident"] = X_val["dayofweek"].isin(
    ["Saturday", "Sunday"]
).astype(int)

X_test["weekend_accident"] = X_test["dayofweek"].isin(
    ["Saturday", "Sunday"]
).astype(int)


# Old vehicle
X_train["old_vehicle"] = X_train["ageofvehicle"].isin(
    ["7 years", "more than 7"]
).astype(int)

X_val["old_vehicle"] = X_val["ageofvehicle"].isin(
    ["7 years", "more than 7"]
).astype(int)

X_test["old_vehicle"] = X_test["ageofvehicle"].isin(
    ["7 years", "more than 7"]
).astype(int)


# High deductible
X_train["high_deductible"] = (X_train["deductible"] > 400).astype(int)
X_val["high_deductible"] = (X_val["deductible"] > 400).astype(int)
X_test["high_deductible"] = (X_test["deductible"] > 400).astype(int)


# Multiple cars
X_train["multiple_cars"] = (X_train["numberofcars"] != "1 vehicle").astype(int)
X_val["multiple_cars"] = (X_val["numberofcars"] != "1 vehicle").astype(int)
X_test["multiple_cars"] = (X_test["numberofcars"] != "1 vehicle").astype(int)

# 7. Encode categoricals

### Ordinal Encoding

In [135]:
ordinal_cols = [
    "vehicleprice",
    "days_policy_accident",
    "pastnumberofclaims",
    "ageofvehicle",
    "ageofpolicyholder",
    "numberofsuppliments",
    "addresschange_claim",
    "numberofcars"
]


In [136]:
ord_enc = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train[ordinal_cols] = ord_enc.fit_transform(X_train[ordinal_cols])
X_val[ordinal_cols] = ord_enc.transform(X_val[ordinal_cols])
X_test[ordinal_cols] = ord_enc.transform(X_test[ordinal_cols])

### Label Encoding

In [137]:
label_cols = [
    "accidentarea",
    "sex",
    "fault",
    "policereportfiled",
    "witnesspresent",
    "agenttype",
    "basepolicy"
]Scale numerical variables


In [138]:
label_encoders = {}

for col in label_cols:
    le = LabelEncoder()

    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_val[col] = le.transform(X_val[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

    label_encoders[col] = le

### One-hot (nominal)


In [139]:
onehot_cols = [
    "make",
    "policytype",
    "vehiclecategory",
    "month",
    "dayofweek",
    "maritalstatus"
]

In [140]:
# Train
X_train = pd.get_dummies(X_train, columns=onehot_cols, drop_first=True)

# Val/Test
X_val = pd.get_dummies(X_val, columns=onehot_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=onehot_cols, drop_first=True)

# Align columns (CRUCIAL)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# 8. Scale numerical variables

In [147]:
num_cols = ["weekofmonth", "age", "deductible", "driverrating", "year"]
scaler = StandardScaler()

# Fit on TRAIN only
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# Apply to val and test
X_val[num_cols] = scaler.transform(X_val[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train.head()

,weekofmonth,accidentarea,sex,age,fault,vehicleprice,deductible,driverrating,days_policy_accident,pastnumberofclaims,...,month_Sep,dayofweek_Monday,dayofweek_Saturday,dayofweek_Sunday,dayofweek_Thursday,dayofweek_Tuesday,dayofweek_Wednesday,maritalstatus_Married,maritalstatus_Single,maritalstatus_Widow
7867,1.727909,1,1,-1.029873,1,1.0,-0.172803,-1.337382,3.0,1.0,...,False,False,False,False,True,False,False,True,False,False
13654,0.167609,1,1,-1.029873,0,4.0,-0.172803,0.453093,3.0,0.0,...,False,False,False,False,True,False,False,True,False,False
8819,-1.392692,1,1,0.687445,0,1.0,-0.172803,1.348330,3.0,1.0,...,False,True,False,False,False,False,False,True,False,False
723,0.947759,1,1,-0.539210,0,0.0,-0.172803,0.453093,3.0,0.0,...,False,False,False,False,False,False,True,False,True,False
14853,0.947759,1,1,0.932776,0,0.0,-0.172803,1.348330,3.0,1.0,...,False,False,True,False,False,False,False,True,False,False


# Resampling


### No Resampling

### Random Oversampling

### SMOTE